# CSCI 4360 Homework 2

### Name:

**Due Date**: go to eLC, click "Tools"->"Assignments" to check for the deadline.

**Submission**: submit this completed Jupyter Notebook file (.ipynb extension) to eLC.

This assignment is to be completed individually. You may use generative AI technology to help you complete this assignment, but you are ultimately responsible for the correctness of your submission.

## Introduction

In this homework assignment you will compare PCA, Gaussian Random Projection, Isomap, LLE, and t-SNE using the Fashion-MNIST dataset. You will evaluate methods based on visualization quality, classification performance, neighborhood preservation, and computational cost.

## Load Fashion-MNIST

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection
from sklearn.manifold import Isomap, LocallyLinearEmbedding, TSNE, trustworthiness
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Fetch Fashion-MNIST dataset from OpenML
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')

# Extract features and targets
X = fashion_mnist.data.astype(np.float32)
y = fashion_mnist.target.astype(np.int64)

print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))

In [ ]:
class_names = [
    "T-shirt/Top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle Boot"
]

fig, axes = plt.subplots(2,5, figsize=(10,5))
for ax, img, label in zip(axes.ravel(), X[:10], y[:10]):
    ax.imshow(img.reshape(28,28), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()

## Part 1: Exploratory Analysis (10 pts)

1. Describe the dataset.
2. Explain why dimensionality reduction may be useful.
3. What classes do you expect to be difficult to separate?

## Helper Functions (Provided)

In [ ]:
def plot_embedding(X_emb, labels, title):
    plt.figure(figsize=(7,6))
    plt.scatter(X_emb[:,0], X_emb[:,1], c=labels, cmap="tab10", s=4, alpha=0.7)
    plt.title(title)
    plt.show()

def evaluate_knn(X_reduced, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X_reduced, y, test_size=0.3, stratify=y, random_state=42
    )

    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train, y_train)

    return accuracy_score(y_test, model.predict(X_test))

## Part 2: Visualization Study (25 pts)

Use a 5,000-sample subset for visualization.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

rng = np.random.RandomState(42)
idx = rng.choice(len(X_scaled), 5000, replace=False)
X_vis = X_scaled[idx]
y_vis = y[idx]

In [ ]:
methods = {
    "PCA": PCA(n_components=2),
    "Random Projection": GaussianRandomProjection(n_components=2, random_state=42),
    "Isomap": Isomap(n_components=2),
    "LLE": LocallyLinearEmbedding(n_components=2, n_neighbors=15),
    "t-SNE": TSNE(n_components=2, random_state=42)
}

embeddings = {}

for name, model in methods.items():
    print(f"Running {name}...")
    emb = model.fit_transform(X_vis)
    embeddings[name] = emb
    plot_embedding(emb, y_vis, name)

### Questions
1. Which method produces the clearest clusters?
2. Which classes overlap the most?
3. Do manifold learning methods outperform PCA visually?

## Part 3: PCA Variance Analysis (10 pts)

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled[:10000])

cumulative = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(7,4))
plt.plot(cumulative)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True)
plt.show()

# TODO: Write Code to determine the EXACT number of dimensions required to preserve 90%, 95%, and 99% variance. Do not just give rough estimates based on the plot.

### Questions
1. What exact number of dimensions is required to preserve 90% variance?
1. What exact number of dimensions is required to preserve 95% variance?
1. What exact number of dimensions is required to preserve 99% variance?

## Part 4: Classification-Based Evaluation (20 pts)

In [ ]:
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_scaled), 10000, replace=False)
X_eval = X_scaled[sample_idx]
y_eval = y[sample_idx]

methods = {
    "PCA": PCA(n_components=50),
    "Random Projection": GaussianRandomProjection(n_components=50, random_state=42),
    "Isomap": Isomap(n_components=50),
    "LLE": LocallyLinearEmbedding(n_components=50, n_neighbors=15)
}

results=[]

for name, model in methods.items():
    start=time.perf_counter()
    X_red=model.fit_transform(X_eval)
    runtime=time.perf_counter()-start
    acc=evaluate_knn(X_red,y_eval)
    results.append([name,acc,runtime])

pd.DataFrame(results, columns=["Method","Accuracy","Runtime (s)"]).sort_values("Accuracy", ascending=False)

### Questions
1. Which method produced the best knn accuracy?
2. Which method takes the least amount of time?
3. Comment on the overall accuracy and runtime trade-offs among the 4 methods.

## Part 5: Trustworthiness Analysis (10 pts)

Trustworthiness in dimensionality reduction is a data quality metric that measures how accurately a low-dimensional embedding preserves the local neighborhood structure of the original high-dimensional space without introducing false neighbors.

In [ ]:
trust_scores=[]

for name, emb in embeddings.items():
    score=trustworthiness(X_vis, emb)
    trust_scores.append([name, score])

pd.DataFrame(trust_scores, columns=["Method","Trustworthiness"]).sort_values("Trustworthiness", ascending=False)

### Questions
1. Which method produced the best trustworthiness?
2. Which method produced the worst trustworthiness?
3. Comment on the relationship between the trustworthiness scores and the plots generated in Part 2: Visualization Study.

## Part 6: Dimensionality vs Accuracy (25 pts)

In Part 4, sample code was provided to compare knn accuracies of the various dimentionality reduction methods when dimensions are reduced to 50 using a subset of 10000 instances. Now expand the experiment to make similar comparisions when dimensions are reduced to 2, 5, 10, 20, 50, and 100. Be sure to create a single line chart comparing knn accuracies for different reduced dimension sizes across all methods.

In [ ]:
dimensions=[2,5,10,20,50,100]

# Complete this experiment for PCA, Random Projection, Isomap, and LLE
# Create a single line chart comparing knn accuracies for different reduced dimension sizes across all methods.

### Questions
1. Commen on knn accuracies for different reduced dimension sizes across all methods.
2. Commen on running times for different reduced dimension sizes acorss all methods.
3. Which dimensionality reduction method would you choose for this dataset, and what would be a good reduced dimension size? Also Comment on the overall accuracy and running time trade-off.